In [1]:
import pandas as pd
import kagglehub
import torch
import torch.nn as nn

from transformers import AutoTokenizer, AutoModel, AutoConfig
from torch.utils.data import Dataset, DataLoader

In [3]:
#Get configs
import json
with open('config.json') as f:
    config = json.load(f)
checkpoint = config['checkpoint']

In [ ]:
#Import scaler
import joblib
scaler = joblib.load(f'scaler_fold_{config['best_fold']}.pkl')

In [ ]:
#Get test.csv
path = kagglehub.competition_download(config['competition'])
test_df = pd.read_csv(f'{path}/test.csv')

In [ ]:
test_df['word_count'] = test_df['full_text'].str.split(' ').str.len()

test_df['num_sentences'] = test_df['full_text'].apply(lambda x: len([s for s in re.split(r'[.!?]+', str(x)) if s.strip()]))

test_df['sentence_length'] = test_df[['num_sentences', 'word_count']].apply(lambda row: row['word_count']/row['num_sentences'], axis=1)

test_df['num_para'] = test_df['full_text'].apply(lambda x: len([p for p in str(x).split('\n') if p.strip()]))

test_df['para_length'] = test_df [['num_para', 'word_count']].apply(lambda row: row['word_count']/row['num_para'], axis=1)

test_df['long_essay'] = (test_df['word_count']>2500).astype(int)

In [ ]:
feature_cols = ['word_count', 'num_sentences', 'sentence_length', 'num_para', 'para_length', 'long_essay']

In [ ]:
class EssayDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=config['max_length']):
        self.X = data['full_text']
        self.features = data[feature_cols]
        if 'score' in data.columns:
            self.y = data['score']
        else:
            self.y = None
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        tokens = self.tokenizer(
            text = self.X.iloc[idx],
            max_length=self.max_length,
            truncation = True,
            padding = 'max_length',
            return_tensors = 'pt'
        )
        item = {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'features': torch.tensor(self.features.iloc[idx].values, dtype=torch.float)
        }
        if self.y is not None:
            item['labels'] = torch.tensor(self.y.iloc[idx], dtype=torch.long)
        return item

In [ ]:
#MeanPooling code taken from online resources
class MeanPooling(nn.Module):
    def __init__(self):
        super(MeanPooling, self).__init__()

    def forward(self, last_hidden_state, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_embeddings/sum_mask
        return mean_embeddings

In [ ]:
class EssayModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained(config['checkpoint'])
        self.mp = MeanPooling()
        #MLP NN
        self.classifier = nn.Sequential(
            nn.Linear(self.model.config.hidden_size + num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )

    def forward(self, input_ids, attention_mask):
        output = self.model(input_ids = input_ids, attention_mask = attention_mask)
        mp_output = self.mp(output.last_hidden_state, attention_mask)
        #Add feature engineered outputs
        combined_output = torch.cat([mp_output, features], dim=1)
        logits = self.classifier(combined_output)       
        return logits

In [ ]:
#Tokenizer
tokenizer = AutoTokenizer.from_pretrained(config['checkpoint'])

#Device
device = torch.device('mps')
#CUDA:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Test
test_df = test_df.copy()
test_df[feature_cols] = scaler.transform(test_df[feature_cols])
test_dataset = EssayDataset(test_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle = False)

#Load best model
deberta_model = EssayModel(num_features=len(feature_cols)).to(device).float()
deberta_model.load_state_dict(
    torch.load(f'best_deberta_model_fold_{config['best_fold']}.pth', map_location=device)
)

#Inference
deberta_model.eval()
test_preds = []

with torch.no_grad():
    for batch in test_loader:
        logits = deberta_model(
            batch['input_ids'].to(device), 
            batch['attention_mask'].to(device),
            batch['features'].to(device)
        )
        preds = torch.round(logits).clamp(0, 5).long()
        test_preds.extend(preds.cpu().numpy())

#Add 1 back to scores
test_preds = [pred+1 for pred in test_preds]

#Submission
submission = pd.DataFrame({
    'essay_id': test_df['essay_id'],
    'score': test_preds
})
submission.to_csv('submission.csv', index=False)